In [1]:
import pandas as pd
import json

print("Running Investment Planning & CAPEX Optimization Engine...")

# 1. Planning FX Rate & Financial Parameters (Naira per USD)
FX_RATE_NGN_PER_USD = 1370  # Standard planning rate benchmark

# 2. Define standard engineering intervention unit costs (USD)
# Based on utility benchmarks (NREL ATB / African utility cost reference data)
UNIT_COSTS_USD = {
    'XFMR_UPGRADE_800KVA': 35000,    # Upgrade distribution transformer to 800kVA
    'XFMR_UPGRADE_500KVA': 22000,    # Upgrade to 500kVA
    'CAPACITOR_BANK_2MVAR': 28000,   # 2 MVAr switched capacitor bank at midpoint
    'RECONDUCTOR_KM': 25000,         # Cost per km to reconductor AAC to ACSR
    'LV_SPLIT_PHASE': 8000           # LV load rebalancing and feeder split
}

# 3. Candidate Projects matched to our EPRI J1 / AEDC Feeder constraints
candidate_projects = [
    {
        "Project_ID": "INV-01",
        "Target_Asset": "TX-07 (Karu Market Spur)",
        "Constraint": "Critical Overload (112%) & Voltage 0.89 pu",
        "Intervention": "Upgrade to 800kVA Transformer",
        "CAPEX_USD": UNIT_COSTS_USD['XFMR_UPGRADE_800KVA'],
        "Customers_Benefited": 160,
        "Expected_Loss_Reduction_kW": 15.5,
        "Load_Relief_pct": 42
    },
    {
        "Project_ID": "INV-02",
        "Target_Asset": "TX-03 (Old Garki Spur)",
        "Constraint": "Critical Overload (105%) & Voltage 0.91 pu",
        "Intervention": "Upgrade to 800kVA Transformer",
        "CAPEX_USD": UNIT_COSTS_USD['XFMR_UPGRADE_800KVA'],
        "Customers_Benefited": 145,
        "Expected_Loss_Reduction_kW": 12.0,
        "Load_Relief_pct": 35
    },
    {
        "Project_ID": "INV-03",
        "Target_Asset": "F-01 Feeder Midpoint",
        "Constraint": "End-of-line Voltage Drop (<0.93 pu)",
        "Intervention": "Install 2MVAr Capacitor Bank",
        "CAPEX_USD": UNIT_COSTS_USD['CAPACITOR_BANK_2MVAR'],
        "Customers_Benefited": 450,
        "Expected_Loss_Reduction_kW": 8.4,
        "Load_Relief_pct": 5
    },
    {
        "Project_ID": "INV-04",
        "Target_Asset": "TX-06 (Area 11 Commercial)",
        "Constraint": "High Load Warning (92%)",
        "Intervention": "Upgrade to 500kVA Transformer",
        "CAPEX_USD": UNIT_COSTS_USD['XFMR_UPGRADE_500KVA'],
        "Customers_Benefited": 120,
        "Expected_Loss_Reduction_kW": 6.8,
        "Load_Relief_pct": 25
    },
    {
        "Project_ID": "INV-05",
        "Target_Asset": "F-01 Trunk Line (2.4 km)",
        "Constraint": "Thermal Line Bottleneck",
        "Intervention": "Reconductor 2.4km AAC to ACSR",
        "CAPEX_USD": UNIT_COSTS_USD['RECONDUCTOR_KM'] * 2.4,
        "Customers_Benefited": 500,
        "Expected_Loss_Reduction_kW": 14.5,
        "Load_Relief_pct": 15
    },
    {
        "Project_ID": "INV-06",
        "Target_Asset": "TX-02 / TX-04 Feeder",
        "Constraint": "Phase Current Imbalance",
        "Intervention": "LV Split Feeder & Phase Balancing",
        "CAPEX_USD": UNIT_COSTS_USD['LV_SPLIT_PHASE'],
        "Customers_Benefited": 45,
        "Expected_Loss_Reduction_kW": 4.2,
        "Load_Relief_pct": 20
    }
]

# 4. Compute Financials & Priority Scores
df_invest = pd.DataFrame(candidate_projects)

# Convert CAPEX to NGN (rounded to nearest ₦100,000)
df_invest['CAPEX_NGN'] = df_invest['CAPEX_USD'].apply(
    lambda x: round((x * FX_RATE_NGN_PER_USD) / 100000) * 100000
)

# Calculate Impact Score = (Customers * 0.4) + (Loss_Reduction_kW * 2.5) + (Load_Relief_pct * 0.8)
df_invest['Priority_Score'] = (
    (df_invest['Customers_Benefited'] * 0.4) + 
    (df_invest['Expected_Loss_Reduction_kW'] * 2.5) + 
    (df_invest['Load_Relief_pct'] * 0.8)
).round(1)

# Sort by Priority Score (Highest engineering ROI first)
df_invest = df_invest.sort_values(by='Priority_Score', ascending=False).reset_index(drop=True)
df_invest['Rank'] = df_invest.index + 1

# Display summary table
display(df_invest[['Rank', 'Project_ID', 'Target_Asset', 'Intervention', 'CAPEX_NGN', 'Priority_Score', 'Customers_Benefited']])

# ==========================================
# EXPORT TO JSON
# ==========================================
export_path = r"C:\Users\Smarterise PC\Projects\PoC\data\investment_register.json"
df_invest.to_json(export_path, orient="records", indent=4)

print(f"\n✅ Investment Portfolio exported successfully to:\n{export_path}")

Running Investment Planning & CAPEX Optimization Engine...


,Rank,Project_ID,Target_Asset,Intervention,CAPEX_NGN,Priority_Score,Customers_Benefited
0,1,INV-05,F-01 Trunk Line (2.4 km),Reconductor 2.4km AAC to ACSR,82200000,248.2,500
1,2,INV-03,F-01 Feeder Midpoint,Install 2MVAr Capacitor Bank,38400000,205.0,450
2,3,INV-01,TX-07 (Karu Market Spur),Upgrade to 800kVA Transformer,48000000,136.4,160
3,4,INV-02,TX-03 (Old Garki Spur),Upgrade to 800kVA Transformer,48000000,116.0,145
4,5,INV-04,TX-06 (Area 11 Commercial),Upgrade to 500kVA Transformer,30100000,85.0,120
5,6,INV-06,TX-02 / TX-04 Feeder,LV Split Feeder & Phase Balancing,11000000,44.5,45



✅ Investment Portfolio exported successfully to:
C:\Users\Smarterise PC\Projects\PoC\data\investment_register.json
